In [1]:
!pip install -q skyfield gradio numpy pandas matplotlib

from datetime import datetime, timedelta, timezone
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from skyfield.api import EarthSatellite, load, wgs84

# Public-format sample TLEs (users can paste their own current TLEs from public sources)
SAMPLE_TLES = """ISS (ZARYA)
1 25544U 98067A   26328.41250000  .00015231  00000-0  27145-3 0  9994
2 25544  51.6392 142.8810 0005124  84.2150 275.9410 15.50123456489123
NOAA 19
1 33591U 09005A   26328.31890000  .00000184  00000-0  13120-3 0  9991
2 33591  99.1840 218.4120 0014210 112.6400 247.6100 14.12456789891234"""


def parse_tle_list(text):
  lines = [ln.strip() for ln in (text or "").strip().splitlines() if ln.strip()]
  sats = []
  i = 0
  while i + 1 < len(lines):
    if (
        not lines[i].startswith("1 ")
        and lines[i + 1].startswith("1 ")
        and (i + 2) < len(lines)
        and lines[i + 2].startswith("2 ")
    ):
      sats.append((lines[i], lines[i + 1], lines[i + 2]))
      i += 3
    elif lines[i].startswith("1 ") and (i + 1) < len(lines) and lines[i + 1].startswith("2 "):
      sats.append((f"SAT-{lines[i][2:7].strip()}", lines[i], lines[i + 1]))
      i += 2
    else:
      i += 1
  return sats


def build_ground_station_dashboard(
    station_name, lat_deg, lon_deg, elev_m, min_el_deg, horizon_hr, tle_text
):
  sats_raw = parse_tle_list(tle_text)
  if not sats_raw:
    err_html = """
        <div style="width:100%;box-sizing:border-box;padding:16px;border-radius:12px;background:#161b26;color:#fbbf24;border:1px solid #b45309;font-family:Inter,sans-serif;">
          No valid TLE entries found. Paste public TLE lines (name + Line 1 + Line 2) or click <b>Load Sample Public TLEs</b>.
        </div>
        """
    return err_html, None, "", pd.DataFrame()

  ts = load.timescale()
  station = wgs84.latlon(float(lat_deg), float(lon_deg), elevation_m=float(elev_m))
  now_utc = datetime.now(timezone.utc)
  end_utc = now_utc + timedelta(hours=float(horizon_hr))

  t0 = ts.from_datetime(now_utc)
  t1 = ts.from_datetime(end_utc)

  pass_rows = []
  max_el_vals = []
  pass_labels = []

  for name, l1, l2 in sats_raw[:4]:
    sat = EarthSatellite(l1, l2, name, ts)
    t_ev, events = sat.find_events(station, t0, t1, altitude_degrees=float(min_el_deg))
    # events: 0=rise (AOS), 1=culmination (max el), 2=set (LOS)
    cur_aos = None
    cur_max_el = None
    for ti, ev in zip(t_ev, events):
      dt_str = ti.utc_datetime().strftime("%Y-%m-%d %H:%M")
      if ev == 0:
        cur_aos = ti.utc_datetime()
      elif ev == 1:
        diff = (sat - station).at(ti)
        alt_d, _, _ = diff.altaz()
        cur_max_el = float(alt_d.degrees)
      elif ev == 2 and cur_aos is not None:
        los_dt = ti.utc_datetime()
        dur_min = max(0.5, (los_dt - cur_aos).total_seconds() / 60.0)
        peak_el = cur_max_el if cur_max_el is not None else float(min_el_deg)
        pass_rows.append({
            "Satellite": name,
            "AOS (UTC)": cur_aos.strftime("%Y-%m-%d %H:%M"),
            "LOS (UTC)": dt_str,
            "Duration (min)": round(dur_min, 1),
            "Max Elevation (deg)": round(peak_el, 1),
        })
        if len(max_el_vals) < 12:
          max_el_vals.append(peak_el)
          pass_labels.append(f"{name[:10]} #{len(max_el_vals)}")
        cur_aos = None
        cur_max_el = None

  # Elevation profile across next 6 hours for primary satellite
  sat0 = EarthSatellite(sats_raw[0][1], sats_raw[0][2], sats_raw[0][0], ts)
  mins_arr = np.linspace(0, min(float(horizon_hr), 24.0) * 60.0, 240)
  dt_arr = [now_utc + timedelta(minutes=float(m)) for m in mins_arr]
  t_prof = ts.utc(
      [d.year for d in dt_arr],
      [d.month for d in dt_arr],
      [d.day for d in dt_arr],
      [d.hour for d in dt_arr],
      [d.minute for d in dt_arr],
      [d.second for d in dt_arr],
  )
  el_prof = (sat0 - station).at(t_prof).altaz()[0].degrees

  plt.close("all")
  plt.style.use("dark_background")
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 4.6), dpi=110)
  fig.patch.set_facecolor("#0b0f17")

  ax1.plot(mins_arr / 60.0, el_prof, color="#38bdf8", linewidth=2.0, label=sat0.name)
  ax1.axhline(
      float(min_el_deg),
      color="#fbbf24",
      linestyle="--",
      linewidth=1.3,
      label=f"Min Elevation Mask ({float(min_el_deg):.0f}°)",
  )
  ax1.axhline(0.0, color="#64748b", linestyle=":", linewidth=1.0)
  ax1.set_title(
      f"Elevation Angle at {station_name}",
      color="#f8fafc",
      fontsize=11,
      fontweight="bold",
  )
  ax1.set_xlabel("Hours From Now (UTC)", color="#94a3b8", fontsize=9)
  ax1.set_ylabel("Elevation (deg)", color="#94a3b8", fontsize=9)
  ax1.tick_params(colors="#94a3b8", labelsize=8)
  ax1.grid(True, alpha=0.18, color="#64748b")
  ax1.legend(loc="upper right", fontsize=8, facecolor="#121826", edgecolor="#263248")
  ax1.set_facecolor("#111724")

  if max_el_vals:
    bars = ax2.bar(pass_labels, max_el_vals, color="#34d399", width=0.55)
    ax2.set_title(
        "Peak Elevation per Detected Pass",
        color="#f8fafc",
        fontsize=11,
        fontweight="bold",
    )
    ax2.set_ylabel("Max Elevation (deg)", color="#94a3b8", fontsize=9)
    ax2.tick_params(axis="x", rotation=25, colors="#94a3b8", labelsize=8)
    ax2.tick_params(axis="y", colors="#94a3b8", labelsize=8)
    for b, v in zip(bars, max_el_vals):
      ax2.text(
          b.get_x() + b.get_width() / 2.0,
          v + 1.0,
          f"{v:.0f}°",
          ha="center",
          va="bottom",
          color="#f8fafc",
          fontsize=8,
          fontweight="bold",
      )
  else:
    ax2.text(
        0.5,
        0.5,
        "No passes above mask in selected window",
        ha="center",
        va="center",
        color="#94a3b8",
        fontsize=10,
    )
    ax2.set_title("Peak Elevation per Pass", color="#f8fafc", fontsize=11, fontweight="bold")

  ax2.grid(True, axis="y", alpha=0.18, color="#64748b")
  ax2.set_facecolor("#111724")
  plt.tight_layout(pad=1.5)

  total_passes = len(pass_rows)
  avg_dur = (
      np.mean([r["Duration (min)"] for r in pass_rows]) if pass_rows else 0.0
  )
  best_el = (
      max([r["Max Elevation (deg)"] for r in pass_rows]) if pass_rows else 0.0
  )

  cards_html = f"""
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;color:#e2e8f0;">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:10px;background:#121826;border:1px solid #263248;border-radius:12px;padding:14px 18px;margin-bottom:12px;">
        <div>
          <div style="font-size:11px;color:#38bdf8;font-weight:700;letter-spacing:0.5px;">PUBLIC-DATA GROUND-STATION PASS MONITOR</div>
          <div style="font-size:20px;font-weight:700;color:#f8fafc;margin-top:2px;">{station_name} <span style="font-size:13px;color:#94a3b8;">({float(lat_deg):.3f}°N, {float(lon_deg):.3f}°E • {float(elev_m):.0f} m)</span></div>
        </div>
        <span style="background:#0f293a;color:#38bdf8;border:1px solid #0284c7;padding:6px 12px;border-radius:999px;font-size:12px;font-weight:700;">Mask ≥ {float(min_el_deg):.0f}° • {float(horizon_hr):.0f}h Window</span>
      </div>

      <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px;">
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">TRACKED SATS</div>
          <div style="font-size:19px;font-weight:700;color:#38bdf8;margin-top:4px;">{len(sats_raw[:4])}</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">DETECTED PASSES</div>
          <div style="font-size:19px;font-weight:700;color:#34d399;margin-top:4px;">{total_passes}</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">AVG PASS DURATION</div>
          <div style="font-size:19px;font-weight:700;color:#a78bfa;margin-top:4px;">{avg_dur:.1f} min</div>
        </div>
        <div style="background:#131a29;border:1px solid #243044;border-radius:10px;padding:12px;">
          <div style="font-size:11px;color:#94a3b8;">PEAK ELEVATION</div>
          <div style="font-size:19px;font-weight:700;color:#fbbf24;margin-top:4px;">{best_el:.1f}°</div>
        </div>
      </div>
    </div>
    """

  guide_html = """
    <div style="width:100%;box-sizing:border-box;font-family:Inter,system-ui,sans-serif;background:#111724;border:1px solid #263248;border-radius:12px;padding:16px 20px;line-height:1.6;font-size:14px;color:#cbd5e1;">
      <div style="font-weight:700;font-size:15px;color:#f8fafc;margin-bottom:8px;">📘 Student Follow-Along Guide (Legal Public Satellite Data Workflow)</div>
      <ol style="margin:0;padding-left:20px;">
        <li><b>Get Legal Public Orbital Data:</b> Copy current Two-Line Element (TLE) sets from an authorized public catalog (such as CelesTrak or official agency open data) and paste them into the TLE box.</li>
        <li><b>Set Ground-Station Coordinates &amp; Mask:</b> Enter your station latitude, longitude, elevation (m), and minimum elevation angle (e.g., <b>15°–20°</b> to account for terrain/buildings/trees).</li>
        <li><b>Select Forecast Horizon &amp; Run:</b> Choose the forecast window (e.g., 24–72 hours) and click <b>Build Ground-Station Dashboard</b> to inspect AOS/LOS times, pass duration, and peak elevation.</li>
        <li><b>Verify &amp; Document:</b> Compare high-elevation passes (&gt; 30°–40°) for stronger line-of-sight geometry, and log results using lawful public telemetry/imagery sources without unauthorized signal interception.</li>
      </ol>
    </div>
    """

  df_out = pd.DataFrame(pass_rows) if pass_rows else pd.DataFrame(
      columns=["Satellite", "AOS (UTC)", "LOS (UTC)", "Duration (min)", "Max Elevation (deg)"]
  )
  return cards_html, fig, guide_html, df_out


full_width_css = """
html, body, .gradio-container, .gradio-container > .main, .gradio-container > .main > .wrap, .block, .form {
  background-color: #0b0f17 !important;
  color: #e2e8f0 !important;
  max-width: 100% !important;
  width: 100% !important;
  margin: 0 !important;
  box-sizing: border-box !important;
}
.gradio-container {
  padding: 10px 14px !important;
}
.table-wrap, .dataframe {
  width: 100% !important;
  max-width: 100% !important;
  overflow-x: visible !important;
}
"""

with gr.Blocks() as demo:
  gr.Markdown("## 📡 Public-Data Ground-Station Dashboard & Student Guide")

  with gr.Row(equal_height=False):
    with gr.Column(scale=4, min_width=320):
      st_name = gr.Textbox(value="Kozan Ground Station", label="Ground Station Name")
      lat_in = gr.Number(value=37.455, label="Station Latitude (deg N)")
      lon_in = gr.Number(value=35.815, label="Station Longitude (deg E)")
      elev_in = gr.Number(value=220.0, label="Station Elevation (m)")
      mask_sl = gr.Slider(5, 60, value=15, step=5, label="Min Elevation Mask (deg)")
      win_sl = gr.Slider(12, 120, value=48, step=6, label="Forecast Window (Hours)")
      tle_in = gr.Textbox(
          value="",
          lines=7,
          label="Paste Legal Public TLE Data",
          placeholder="Paste 2-line or 3-line TLE blocks or click Load Sample Public TLEs...",
      )
      sample_btn = gr.Button("📄 Load Sample Public TLEs")
      run_btn = gr.Button("⚡ Build Ground-Station Dashboard", variant="primary")

    with gr.Column(scale=8, min_width=460):
      out_cards = gr.HTML()
      out_plot = gr.Plot(label="Elevation Profile & Pass Peak Angles")

  with gr.Row():
    with gr.Column(scale=12):
      out_guide = gr.HTML()
      out_table = gr.Dataframe(
          label="Predicted Ground-Station Passes (UTC)", interactive=False
      )

  sample_btn.click(fn=lambda: SAMPLE_TLES, inputs=[], outputs=[tle_in])
  run_btn.click(
      fn=build_ground_station_dashboard,
      inputs=[st_name, lat_in, lon_in, elev_in, mask_sl, win_sl, tle_in],
      outputs=[out_cards, out_plot, out_guide, out_table],
  )

demo.launch(share=False, theme=gr.themes.Soft(), css=full_width_css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>